In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt

rng_seed = 42  
n_random_lines = 30

In [ ]:
obs_id = "m3g20090617t045633"

In [ ]:
def bde_correction(obs_data: np.ndarray, bde_map: np.ndarray):
    """
    Elements are flagged 0-5 in these fits files. Values 1-4 seem to indicate
    things that are "flagged", 1 being the lowest level and 4 the highest. I
    think the mission interpolated everything flagged 1-4 in the spectral
    direction. This does seem a little problematic to me because the filter
    seams (channel features) are flagged, usually as 4, so right now we
    interpolate across them twice.

    Pixels with only one "good" pixel above or below it (this means they are
    likely near the edge) will adopt that good pixel's value. If there are NO
    good pixels in a column, right now we don't do anything. This could change,
    but should only matter for the tap columns we interpolate across in the
    spatial direction in a later step.

    Supports 3D arrays (frames, channels, cols) and 2D arrays (channels, cols).
    For 3D arrays, correction is applied per-frame using that frame's values.
    """

    bad_mask = bde_map != 0
    n_rows, n_cols = bad_mask.shape  # 86 x 320 for global mode

    # we only need to figure out the interpolation weights once for the obs
    # new_val =
    # val_at_top + (bad_row-top_row)/(bot_row-top_row)*(val_at_bot-val_at_top)
    # where the "weight" is (bad_row-top_row)/(bot_row-top_row)

    bad_rows, bad_cols = np.where(bad_mask)  # all bad pixels

    # find the closest top and bottom pixel
    top_rows = np.full(len(bad_rows), -1,
                       dtype=np.intp)  # np.intp is for indices
    bottom_rows = np.full(len(bad_rows), n_rows, dtype=np.intp)

    # "good" pixels per column
    good_in_col = []
    for c in range(n_cols):
        # get all good rows
        good_in_col.append(np.where(~bad_mask[:, c])[0])

    # find closest good pixels
    for i, (r, c) in enumerate(zip(bad_rows, bad_cols)):
        g = good_in_col[c]
        if g.size == 0:
            continue
            # TODO: IDK what to do if the whole column is bad?
            #  for readout cols we interpolate across the col (spatially)
            #  instead of across channels (spectrally)
        idx = np.searchsorted(g, r)
        if idx > 0:
            top_rows[i] = g[idx - 1]  # above
        if idx < len(g):
            bottom_rows[i] = g[idx]  # below

    has_top = top_rows >= 0
    has_bottom = bottom_rows < n_rows
    both = has_top & has_bottom
    top_only = has_top & ~has_bottom
    bottom_only = ~has_top & has_bottom
    # neither = ~has_top & ~has_bottom
    # we ignore this right now,
    # could interpolate across / spatially

    # weights for linear interpolation when there is a top and bottom pixel
    denominator = np.where(both, bottom_rows - top_rows, 1)
    t = np.where(both, (bad_rows - top_rows) / denominator, 0.0)

    is_3d = obs_data.ndim == 3

    if both.any():
        br = bad_rows[both]
        bc = bad_cols[both]
        tr = top_rows[both]
        btr = bottom_rows[both]
        w = t[both]

        if is_3d:
            n_frames = obs_data.shape[0]
            for frame_idx in range(n_frames):
                top_vals = obs_data[frame_idx, tr, bc]
                bot_vals = obs_data[frame_idx, btr, bc]
                interp = top_vals + w * (bot_vals - top_vals)
                obs_data[frame_idx, br, bc] = interp
        else:
            top_vals = obs_data[tr, bc]
            bot_vals = obs_data[btr, bc]
            interp = top_vals + w * (bot_vals - top_vals)
            obs_data[br, bc] = interp
    if top_only.any():
        br = bad_rows[top_only]
        bc = bad_cols[top_only]
        tr = top_rows[top_only]
        if is_3d:
            n_frames = obs_data.shape[0]
            for frame_idx in range(n_frames):
                obs_data[frame_idx, br, bc] = obs_data[frame_idx, tr, bc]
        else:
            obs_data[br, bc] = obs_data[tr, bc]

    if bottom_only.any():
        br = bad_rows[bottom_only]
        bc = bad_cols[bottom_only]
        btr = bottom_rows[bottom_only]
        if is_3d:
            n_frames = obs_data.shape[0]
            for frame_idx in range(n_frames):
                obs_data[frame_idx, br, bc] = obs_data[frame_idx, btr, bc]
        else:
            obs_data[br, bc] = obs_data[btr, bc]

    return obs_data


In [ ]:
metadata = pd.read_csv("/home/bekah/m3-pipeline-dev/obs_to_cal_file_mapping/obs_cal_info.csv")
metadata = metadata[metadata['obs_id'] == obs_id.upper()]

if len(metadata) == 0:
    print("This is not a valid observation ID, although it could be a dark"
          " signal observation.")
elif len(metadata) > 1:
    best_idx = metadata['version'].str.extract(r'(\d+)')[0].astype(int).idxmax()
    metadata = metadata.loc[[best_idx]]

dark_id = metadata['dark_signal_id'].iloc[0].lower()
flat_id = metadata['flat_field_id'].iloc[0].lower()
bde_id = metadata['bad_detector_map_id'].iloc[0].lower()

print(f"Temp: {metadata['obs_temperature'].iloc[0]}")
print(f"Dark ID: {dark_id}")
print(f"Flat ID: {flat_id}")
print(f"BDE ID: {bde_id}")

dark_path = f"/home/bekah/m3-pipeline-dev/data/dark/darks_global/{dark_id}_l0.fits"
with fits.open(dark_path) as hdul:
    dark = hdul[0].data.transpose(1, 0, 2)[5:-5, :, :]
dark = np.median(dark, axis=0)

obs_path = f"/home/bekah/m3-pipeline-dev/matching_area_obs/{obs_id}_l0.fits"
with fits.open(obs_path) as hdul:
    image = hdul[0].data.transpose(1, 0, 2)[5:-5, :, :]

image = image - dark[np.newaxis, :, :]

# load bde
obs_path = f"/home/bekah/m3-pipeline-dev/data/bde/m3g/{bde_id}_bde.fits"
with fits.open(obs_path) as hdul:
    header = hdul[0].header.copy()
    bde = hdul[0].data
    
# apply bde to obs 
image = bde_correction(image, bde) 

image = image.transpose(1, 0, 2)  # -> (band, line, sample)
del dark


In [ ]:
n_bands, n_lines, n_samples = image.shape

rng = np.random.default_rng(rng_seed)
random_lines = rng.choice(n_lines, size=n_random_lines, replace=False)
random_lines.sort()

print("lines:", random_lines)

In [ ]:
def line_stats(band_slice_2d, line_idx):

    line = band_slice_2d[line_idx]

    return {
        "median_20_300":   np.mean(line[20:300]),
        "per_20_300": np.percentile(line[20:300], 90),
        "mean_1_3":      np.mean(line[1:3]),
        "mean_318_320":  np.mean(line[318:320]),
        "value_1":       line[1],
        "median_1_4":    np.median(line[1:4]),
    }

records = []
for band_idx in range(n_bands):
    band_2d = image[band_idx] 
    for line_idx in random_lines:
        stats = line_stats(band_2d, line_idx)
        records.append({
            "band": band_idx,
            "line": int(line_idx),
            **stats,
        })

df = pd.DataFrame.from_records(records)


In [ ]:

stat_cols = [c for c in df.columns if c not in ("band", "line")]
lines_sorted = sorted(df["line"].unique())

cmap = plt.get_cmap("viridis")
colors = {ln: cmap(i / max(len(lines_sorted) - 1, 1)) for i, ln in enumerate(lines_sorted)}

fig, axes = plt.subplots(len(stat_cols), 1, figsize=(9, 3 * len(stat_cols)), sharex=True)
for ax, stat in zip(axes, stat_cols):
    for ln in lines_sorted:
        sub = df[df["line"] == ln].sort_values("band")
        ax.plot(sub["band"], sub[stat]/sub['median_20_300'], color=colors[ln], label=f"line {ln}", linewidth=1)
    ax.set_ylabel(stat)
    ax.set_ylim(-.1,.05)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("band")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(1.15, 1), title="line")
fig.suptitle("Stats vs band, colored by line", y=1.02)
fig.tight_layout()
plt.show()